Step 1: Install and Load the Dataset

In [1]:
# pip install datasets tensorflow

from datasets import load_dataset
import re
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Load AG News dataset
dataset = load_dataset("wangrongsheng/ag_news")

train_text = dataset["train"]["text"]
train_label = dataset["train"]["label"]

test_text = dataset["test"]["text"]
test_label = dataset["test"]["label"]



README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Step 2: Clean and Normalize the Text

In [2]:

def clean_text(text):
    text = text.lower()                     # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text)    # Remove numbers & punctuation
    return text

train_text = [clean_text(text) for text in train_text]
test_text = [clean_text(text) for text in test_text]

Step 3: Tokenize the Strings into Numbers

In [3]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_text)

X_train = tokenizer.texts_to_sequences(train_text)
X_test = tokenizer.texts_to_sequences(test_text)



 Step 4: Pad and Truncate Sequences

In [4]:
max_length = 50

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)


Step 5: Convert Labels to Categorical

In [5]:

y_train = to_categorical(train_label, num_classes=4)
y_test = to_categorical(test_label, num_classes=4)


# ==========================================
# Step 6: Define the RNN Model Architecture
# ==========================================

model = Sequential()

model.add(Embedding(input_dim=vocab_size,
                    output_dim=128,
                    input_length=max_length))

model.add(SimpleRNN(64))

model.add(Dense(4, activation='softmax'))



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


==========================================
# Step 6: Define the RNN Model Architecture

In [6]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 60s 38ms/step - accuracy: 0.8236 - loss: 0.5097 - val_accuracy: 0.8717 - val_loss: 0.3762
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 54s 36ms/step - accuracy: 0.8943 - loss: 0.3308 - val_accuracy: 0.8552 - val_loss: 0.4288
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 55s 37ms/step - accuracy: 0.9100 - loss: 0.2848 - val_accuracy: 0.8611 - val_loss: 0.4257
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 55s 36ms/step - accuracy: 0.9203 - loss: 0.2556 - val_accuracy: 0.8633 - val_loss: 0.4072
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 55s 37ms/step - accuracy: 0.9250 - loss: 0.2420 - val_accuracy: 0.8763 - val_loss: 0.3976



Step 7: Compile and Train the Model

In [7]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8911 - loss: 0.3653
Test Loss : 0.3653029203414917
Test Accuracy : 0.8910526037216187
